In [ ]:
from pathlib import Path
import os
from rich import print

import numpy as np
from deep_orderbook.learn.current_branch_learn import train_and_predict, SHAPER
from deep_orderbook.visu import Visualizer
from deep_orderbook.config import TrainConfig, ReplayConfig
from deep_orderbook.strategy import Strategy
from deep_orderbook.strategy_search import evaluate_short_strategy

CWD = Path.cwd().resolve()
# print(CWD)
if CWD.name == 'deep_orderbook':
    REPO_ROOT = CWD
elif CWD.name == 'notebooks' and CWD.parent.name == 'deep_orderbook':
    REPO_ROOT = CWD.parent
else:
    raise RuntimeError(f'Run this notebook from deep_orderbook repo or notebooks/, got {CWD}')

os.chdir(REPO_ROOT)

train_config = TrainConfig(
    num_workers=5,
    batch_size=16,
    data_queue_size=1024,
    num_levels=8,
    learning_rate=0.001,
    epochs=100,
    save_checkpoint_batches=10,
    save_checkpoint_mins=1.0,
)
replay_config = ReplayConfig(
    markets=["BTC-USD"],
    date_regexp='2026-04-1*T2*',
    data_dir='../crypto',
    every="1000ms",
    randomize=True,
)
shaper_config = SHAPER
# print(shaper_config)
test_config = replay_config.but(
    # date_regexp='2026-04-20*',
    randomize=False,
)

DISPLAY_EVERY_BATCHES = 2
MOVE_TEST_WINDOW = True
# Route to display — primary from active-core short research
DISPLAY_ROUTE = "q90_p1_hold48"
# MOVE_TEST_WINDOW walks through filtered test windows in chronological order.


In [ ]:
vis = Visualizer()
gt_strategy = Strategy(threshold=0.10)

async def online_learning(train_config, replay_config, shaper_config):
    async for books_arr, t2l, pxar, pred_t2l, train_loss, test_loss, route_configs in train_and_predict(
        config=train_config, replay_config=replay_config, shaper_config=shaper_config, test_config=test_config,
        display_every_batches=DISPLAY_EVERY_BATCHES,
        move_test_window=MOVE_TEST_WINDOW,
    ):
        # Ground truth PnL with a simple threshold (visual reference only)
        gt_pnl, pos, gt_up_prox, gt_down_prox = gt_strategy.compute_pnl(pxar, t2l)

        # Predicted PnL using the calibrated route from exp63/exp64 pipeline
        # route_configs is empty for the first epoch's intra-batch yields — fall back to gt_strategy
        route = next((r for r in route_configs if r.name == DISPLAY_ROUTE), None)
        if route is not None:
            pred_route = evaluate_short_strategy(pxar, pred_t2l[..., 0], **route.strategy_kwargs())
            pred_pnl = np.asarray(pred_route["pnl"])
            pred_pos = np.asarray(pred_route["positions"])
        else:
            pred_pnl, pred_pos, _, _ = gt_strategy.compute_pnl(pxar, pred_t2l)

        # Proximity display still uses the simple strategy on predictions
        _, _, pred_up_prox, pred_down_prox = gt_strategy.compute_pnl(pxar, pred_t2l)

        vis.add_loss(train_loss, test_loss)
        vis.update(
            books_z_data=books_arr, level_reach_z_data=t2l, bidask=pxar,
            pred_t2l=pred_t2l, gt_pnl=gt_pnl, pred_pnl=pred_pnl,
            positions=pos, pred_positions=pred_pos,
            up_proximity=gt_up_prox, down_proximity=gt_down_prox,
            pred_up_proximity=pred_up_prox, pred_down_proximity=pred_down_prox
        )
await online_learning(train_config=train_config, replay_config=replay_config, shaper_config=shaper_config)
